In [1]:
import multiprocessing

print(multiprocessing.cpu_count())

12


In [2]:
# ============================================================
# HUGGING FACE DATASETS - LESSON 5 COMPLETE PRACTICE NOTEBOOK
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL (Only if required)
# ------------------------------------------------------------
# !pip install -U datasets

# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------

from datasets import (
    load_dataset,
    concatenate_datasets,
    interleave_datasets
)

import multiprocessing

# ------------------------------------------------------------
# 3. CHECK NUMBER OF CPUs
# Useful for parallel processing with map/filter
# ------------------------------------------------------------

num_cpus = multiprocessing.cpu_count()

print(f"Available CPUs: {num_cpus}")

# ============================================================
# 4. LOAD IMDB DATASET
# ============================================================

imdb_dataset = load_dataset("stanfordnlp/imdb")

print(imdb_dataset)

# DatasetDict({
#     train: Dataset(...)
#     test: Dataset(...)
# })

# ------------------------------------------------------------
# View train split
# ------------------------------------------------------------

print(imdb_dataset["train"])

# ------------------------------------------------------------
# Number of samples
# ------------------------------------------------------------

print(len(imdb_dataset["train"]))

# ------------------------------------------------------------
# Dataset schema / features
# ------------------------------------------------------------

print(imdb_dataset["train"].features)

# ------------------------------------------------------------
# View first sample
# ------------------------------------------------------------

print(imdb_dataset["train"][0])

# ------------------------------------------------------------
# View first 5 samples
# ------------------------------------------------------------

print(imdb_dataset["train"][:5])

# ------------------------------------------------------------
# Column names
# ------------------------------------------------------------

print(imdb_dataset["train"].column_names)

# ============================================================
# 5. LOAD COMPLETE IMDB DATASET
# Merge train + test
# ============================================================

imdb_full = load_dataset(
    "stanfordnlp/imdb",
    split="train+test"
)

print(imdb_full)

print(f"Total Samples: {len(imdb_full)}")

# Expected:
# 50000 samples

# ============================================================
# 6. LOAD ROTTEN TOMATOES DATASET
# ============================================================

rt_dataset = load_dataset(
    "cornell-movie-review-data/rotten_tomatoes",
    split="all"
)

print(rt_dataset)

print(rt_dataset.features)

print(f"Total Samples: {len(rt_dataset)}")

# ------------------------------------------------------------
# View sample
# ------------------------------------------------------------

print(rt_dataset[0])

# ------------------------------------------------------------
# Label names
# ------------------------------------------------------------

print(
    rt_dataset.features["label"].names
)

# Expected:
# ['neg', 'pos']

# ============================================================
# 7. FILTER DATASET
# Keep only reviews having >=100 words
# ============================================================

MIN_WORDS = 100

filtered_imdb = imdb_dataset.filter(
    lambda example:
    len(example["text"].split()) >= MIN_WORDS,

    # Parallel processing
    num_proc=num_cpus
)

print(filtered_imdb)

# Compare before and after

print(
    "Original Train:",
    len(imdb_dataset["train"])
)

print(
    "Filtered Train:",
    len(filtered_imdb["train"])
)

# ============================================================
# 8. MAP OPERATION
# Apply transformation to every sample
# ============================================================

def add_prefix(example):

    # Add prefix to every review

    example["text"] = (
        "IMDb Review: "
        + example["text"]
    )

    return example


modified_dataset = imdb_dataset.map(
    add_prefix
)

print(modified_dataset["train"][0])

# ============================================================
# 9. ANOTHER MAP EXAMPLE
# Convert all text to lowercase
# ============================================================

def lowercase_text(example):

    example["text"] = (
        example["text"].lower()
    )

    return example


lower_dataset = imdb_dataset.map(
    lowercase_text
)

print(lower_dataset["train"][0])

# ============================================================
# 10. CONCATENATE DATASETS
# Combine IMDb + Rotten Tomatoes
# ============================================================

combined_dataset = concatenate_datasets(
    [
        imdb_full,
        rt_dataset
    ]
)

print(combined_dataset)

print(
    f"Combined Samples: {len(combined_dataset)}"
)

# Expected:
# 50000 + 10000 = 60000 approx

# ============================================================
# 11. CREATE TRAIN TEST SPLIT
# After concatenation
# ============================================================

split_dataset = (
    combined_dataset
    .train_test_split(
        test_size=0.2,
        shuffle=True,
        seed=42
    )
)

print(split_dataset)

# ============================================================
# 12. INTERLEAVE DATASETS
# Maintain desired proportions
# ============================================================

interleaved_dataset = interleave_datasets(

    [
        imdb_full,
        rt_dataset
    ],

    probabilities=[
        0.6,   # IMDb
        0.4    # Rotten Tomatoes
    ],

    seed=42
)

print(interleaved_dataset)

print(
    f"Interleaved Samples: {len(interleaved_dataset)}"
)

# Note:
# Interleaving stops when one dataset
# runs out of samples (default behavior)

# ============================================================
# 13. STREAMING DATASET
# Load without downloading entire dataset
# ============================================================

imdb_stream = load_dataset(
    "stanfordnlp/imdb",
    split="train",
    streaming=True
)

print(imdb_stream)

# Output:
# IterableDataset

# ============================================================
# 14. ITERATE OVER STREAMING DATASET
# ============================================================

for sample in imdb_stream:

    print(sample)

    break

# Only one sample fetched

# ============================================================
# 15. MAP ON STREAMING DATASET
# Lazy evaluation
# ============================================================

stream_with_prefix = (
    imdb_stream.map(add_prefix)
)

for sample in stream_with_prefix:

    print(sample)

    break

# Processing happens only when
# sample is fetched

# ============================================================
# 16. FILTER ON STREAMING DATASET
# ============================================================

filtered_stream = imdb_stream.filter(

    lambda example:
    len(example["text"].split()) >= 100

)

for sample in filtered_stream:

    print(sample)

    break

# Filter is applied on-the-fly

# ============================================================
# 17. LOAD DATASET FROM EXTERNAL URL
# ============================================================

iris_dataset = load_dataset(

    "csv",

    data_files=
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

)

print(iris_dataset)

print(
    iris_dataset["train"][0]
)

# ============================================================
# 18. LOAD LOCAL CSV FILE
# ============================================================

# dataset = load_dataset(
#     "csv",
#     data_files="students.csv"
# )

# ============================================================
# 19. LOAD JSON FILE
# ============================================================

# dataset = load_dataset(
#     "json",
#     data_files="data.json"
# )

# ============================================================
# 20. LOAD TEXT FILE
# ============================================================

# dataset = load_dataset(
#     "text",
#     data_files="reviews.txt"
# )

# ============================================================
# 21. LOAD PARQUET FILE
# ============================================================

# dataset = load_dataset(
#     "parquet",
#     data_files="train.parquet"
# )

# ============================================================
# 22. LOAD DATASET FROM ZIP FILE
# ============================================================

# dataset = load_dataset(
#     "csv",
#     data_files="dataset.zip"
# )

# ============================================================
# 23. EXTRACT ZIP FILE MANUALLY
# ============================================================

# import zipfile

# with zipfile.ZipFile(
#     "dataset.zip"
# ) as z:
#
#     z.extractall("data")

# dataset = load_dataset(
#     "csv",
#     data_files="data/train.csv"
# )

# ============================================================
# 24. USEFUL COMMANDS FOR ANY DATASET
# ============================================================

# Dataset length
print(len(rt_dataset))

# Features
print(rt_dataset.features)

# Column names
print(rt_dataset.column_names)

# First sample
print(rt_dataset[0])

# First 5 samples
print(rt_dataset[:5])

# Random sample

import random

idx = random.randint(
    0,
    len(rt_dataset)-1
)

print(rt_dataset[idx])

# ============================================================
# END OF LESSON 5
# ============================================================

/home/mypc/Desktop/dlp/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available CPUs: 12


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})
25000
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['neg', 'pos'], id=None)}
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documen

Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 329486.22 examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 10662
})
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['neg', 'pos'], id=None)}
Total Samples: 10662
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
['neg', 'pos']


...

{'text': 'IMDb Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are f

Map: 100%|██████████| 50000/50000 [00:01<00:00, 32890.69 examples/s]


{'text': 'i rented i am curious-yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by u.s. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" i really had to see this for myself.<br /><br />the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life. in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such as the vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />what kills me about i am curious-yellow is that 40 years ago, this was considered pornographic. really, the sex and nudity scenes are few and far be

Generating train split: 150 examples [00:00, 18223.96 examples/s]

DatasetDict({
    train: Dataset({
        features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'],
        num_rows: 150
    })
})
{'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2, 'species': 'setosa'}
10662
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['neg', 'pos'], id=None)}
['text', 'label']
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-

# Interleaving Datasets

## Why Do We Need It?

In real-world machine learning, datasets are often **imbalanced (skewed)**.

For example:

| Dataset | Number of Samples |
|----------|----------:|
| IMDb | 50,000 |
| Rotten Tomatoes | 10,000 |

If we simply combine them:

```python
combined = imdb + rotten_tomatoes
```

the resulting dataset would contain:

```text
IMDb            : 50,000 samples (83.3%)
Rotten Tomatoes : 10,000 samples (16.7%)
```

The model would mostly learn from IMDb and very little from Rotten Tomatoes.

---

## Solution: Interleaving

Instead of appending datasets one after another, we can **interleave** them.

Interleaving means:

> Create a new dataset by alternately selecting samples from multiple datasets according to specified probabilities.

This ensures each dataset contributes a desired proportion of samples.

---

## Example

Suppose:

```text
IMDb            : 50,000 samples
Rotten Tomatoes : 10,000 samples
```

We want:

```text
60% IMDb
40% Rotten Tomatoes
```

The new dataset will approximately follow:

```text
IMDb            → 60%
Rotten Tomatoes → 40%
```

even though IMDb originally contains many more samples.

---

## Visual Comparison

### Simple Concatenation

```text
IMDb IMDb IMDb IMDb IMDb IMDb IMDb IMDb ...
IMDb IMDb IMDb IMDb IMDb IMDb IMDb IMDb ...
Rotten Tomatoes ...
Rotten Tomatoes ...
```

Dataset is dominated by IMDb.

---

### Interleaving (60%-40%)

```text
IMDb
Rotten Tomatoes
IMDb
IMDb
Rotten Tomatoes
IMDb
Rotten Tomatoes
IMDb
IMDb
Rotten Tomatoes
...
```

The sampling follows the desired distribution.

---

## Hugging Face Implementation

```python
from datasets import interleave_datasets

combined_dataset = interleave_datasets(
    [imdb_dataset, rt_dataset],
    probabilities=[0.6, 0.4],
    seed=42
)
```

### Parameters

| Parameter | Meaning |
|------------|----------|
| datasets | List of datasets to combine |
| probabilities | Sampling probability for each dataset |
| seed | Makes sampling reproducible |

---

## Example Calculation

Assume we want a dataset of 100 samples.

Using:

```python
probabilities=[0.6, 0.4]
```

Expected distribution:

```text
IMDb            ≈ 60 samples
Rotten Tomatoes ≈ 40 samples
```

---

## Why Is This Useful?

### 1. Balance Multiple Datasets

```text
Large Dataset  : 1,000,000 samples
Small Dataset  :   50,000 samples
```

Without interleaving, the smaller dataset contributes almost nothing.

---

### 2. Multi-Task Learning

Combine datasets from different tasks:

```text
Sentiment Analysis
Question Answering
Text Classification
```

while controlling their contribution.

---

### 3. Training Large Language Models

Modern LLMs are trained using data from:

```text
Wikipedia
Books
Research Papers
Code
Web Pages
```

Interleaving helps maintain desired proportions.

---

## Key Takeaway

- **Concatenation** simply joins datasets one after another.
- **Interleaving** mixes datasets according to specified probabilities.
- It is especially useful when datasets have very different sizes.
- Example:

```python
interleave_datasets(
    [imdb_dataset, rt_dataset],
    probabilities=[0.6, 0.4]
)
```

creates a dataset where approximately:

```text
60% samples come from IMDb
40% samples come from Rotten Tomatoes
```

regardless of their original sizes.